# FABLE Pyculator 2021 FreshForge Run

This notebook shows how to use FABLE Pyculator to derive the 2021 FABLE-C output-ref boundary, then use FreshForge to run the Modelwright generated-model workflow.

## Local artifacts

The source workbook stays under ignored `tmp/` paths. The generated Python model and run evidence are written under `tmp/generated-models/fable-2021/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

from IPython.display import Markdown, display

from fable_pyculator import (
    build_2021_notebook_spec,
    build_cached_workbook_validation_scenario,
    build_modelwright_freshforge_workflow,
    derive_output_refs,
    freshforge_2021_build_paths,
    load_generated_model,
    run_notebook_loop,
    write_freshforge_workflow,
    write_output_refs,
    write_validation_scenario,
)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "fable_pyculator").exists():
            return candidate
    raise RuntimeError("Could not find the fable-pyculator repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
paths = freshforge_2021_build_paths(repo_root=repo_root)
paths.artifact_dir.mkdir(parents=True, exist_ok=True)

workbook_path = paths.workbook_path
artifact_dir = paths.artifact_dir
output_refs_path = paths.output_refs_path
workflow_path = paths.workflow_path
generated_model_path = paths.generated_model_path
validation_scenario_path = paths.validation_scenario_path
evaluation_report_path = paths.evaluation_report_path

display(Markdown(f"Workbook path: `{workbook_path.relative_to(repo_root)}`"))
display(Markdown(f"Artifact directory: `{artifact_dir.relative_to(repo_root)}`"))


In [ ]:
missing = [path for path in (workbook_path,) if not path.exists()]
ARTIFACTS_READY = not missing

if missing:
    display(Markdown("Missing local artifact(s):"))
    display([str(path.relative_to(repo_root)) for path in missing])
else:
    display(Markdown("Local 2021 workbook is ready."))

## Derive Modelwright output refs

FABLE Pyculator reads the FABLE output tables and selects workbook cells from columns tagged `OUTPUT-*`. Modelwright receives the resulting explicit `output_refs.json` file.

In [ ]:
if ARTIFACTS_READY:
    fable_spec = build_2021_notebook_spec(workbook_path)
    output_refs = derive_output_refs(fable_spec, column_flavour_tags="OUTPUT-*")
    write_output_refs(output_refs_path, output_refs)
    display(Markdown(f"Wrote `{len(output_refs):,}` output refs to `{output_refs_path.relative_to(repo_root)}`."))
    display(output_refs[:10])
else:
    output_refs = []
    display(Markdown("Restore the 2021 workbook before deriving output refs."))


## Prepare cached-workbook validation

The validation scenario uses nonblank cached workbook values for the selected output refs.

In [ ]:
if ARTIFACTS_READY and output_refs:
    validation_scenario = build_cached_workbook_validation_scenario(
        workbook_path,
        output_refs,
        generated_model_path=generated_model_path,
        scenario_id="fable-c-2021-freshforge-run",
        description="Cached-workbook validation slice derived from FABLE Pyculator OUTPUT-* refs.",
        source_workbook=workbook_path.relative_to(repo_root),
        generated_model=generated_model_path.relative_to(repo_root),
    )
    write_validation_scenario(validation_scenario_path, validation_scenario)
    display(Markdown(f"Wrote `{len(validation_scenario['outputs']):,}` validation outputs to `{validation_scenario_path.relative_to(repo_root)}`."))
else:
    display(Markdown("Restore the 2021 workbook and derive output refs before writing a validation scenario."))


## Declare the FreshForge workflow

This graph uses Modelwright provider nodes for contract inference, model generation, generated-model execution, and cached validation evaluation.

In [ ]:
workflow_document = build_modelwright_freshforge_workflow(
    paths,
    workdir=repo_root,
    workflow_id="fable_2021_modelwright_run",
    name="FABLE 2021 Modelwright FreshForge run",
    description="FreshForge graph for rebuilding the 2021 FABLE generated model.",
    module_name="generated_fable_2021_model",
)
write_freshforge_workflow(workflow_path, workflow_document)
display(Markdown(f"Wrote FreshForge workflow graph to `{workflow_path.relative_to(repo_root)}`."))


## Validate and plan

In [ ]:
try:
    from freshforge.loading import load_workflow
    from freshforge.planning import create_run_plan

    FRESHFORGE_READY = True
except ModuleNotFoundError:
    FRESHFORGE_READY = False
    display(Markdown("FreshForge is not installed in this kernel."))

if FRESHFORGE_READY:
    workflow_spec, diagnostics = load_workflow(workflow_path)
    plan = create_run_plan(workflow_spec, diagnostics=diagnostics) if workflow_spec is not None else None
    display(plan.to_dict() if plan is not None else [diagnostic.to_dict() for diagnostic in diagnostics])
    if plan is None or plan.has_errors:
        raise RuntimeError("FreshForge could not plan the Modelwright workflow graph.")

## Run with FreshForge

The run is gated because the full 2021 build can take several minutes. Review the plan first, then set `RUN_FRESHFORGE = True`.

In [ ]:
RUN_FRESHFORGE = False

if RUN_FRESHFORGE:
    if not ARTIFACTS_READY:
        raise RuntimeError("Restore the 2021 workbook before running the build.")
    from freshforge.execution import run_workflow
    from freshforge.loading import load_workflow

    workflow_spec, diagnostics = load_workflow(workflow_path)
    if workflow_spec is None:
        raise RuntimeError([diagnostic.to_dict() for diagnostic in diagnostics])
    run_result = run_workflow(workflow_spec, diagnostics=diagnostics, workdir=repo_root)
    display(run_result.to_dict())
    if not run_result.ok:
        raise RuntimeError("FreshForge run failed; inspect diagnostics above.")
else:
    display(Markdown("FreshForge run not started. Set `RUN_FRESHFORGE = True` after reviewing the plan."))

## Inspect FABLE Pyculator outputs

After the FreshForge run creates the 2021 generated model, load that exact generated artifact and inspect FABLE output tables/headlines.

In [ ]:
if generated_model_path.exists() and ARTIFACTS_READY:
    generated_model = load_generated_model(
        generated_model_path,
        module_name="fable_pyculator_freshforge_generated_fable_2021",
    )
    result = run_notebook_loop(
        generated_model,
        fable_spec,
        {},
        scenario_name="freshforge-2021",
        output_table_column_flavour_tags="OUTPUT-*",
        include_figures=False,
    )
    display(Markdown(f"Rendered `{len(result.output_tables)}` output tables and `{len(result.headline_frames)}` headline frames."))
    display(result.output_tables["ghg_resultsghg"].head())
else:
    display(Markdown(f"Generated model not found yet: `{generated_model_path.relative_to(repo_root)}`"))